# Sigma2 Fitting with Compact Multi-ISI Sequences

Uses `ISISequence` + `StimulusManager` to generate short multi-ISI sequences
(~78 trials) containing interleaved repeat pairs at ISIs 8, 16, 32, 64.
This replaces the per-ISI toy experiments (which are very long for high ISIs)
and avoids using real participant sequences (reserved for final evaluation).

| Section | Purpose |
|---------|----------|
| A | Generate & validate compact multi-ISI sequences |
| B | Sigma2 grid sweep using compact sequences |
| C | Stability: vary n_sequences_per_rep |
| D | Compare to per-ISI toy baseline |

In [ ]:
import sys, os, yaml, torch, random
import matplotlib.pyplot as plt, numpy as np, pandas as pd
from collections import defaultdict, Counter
from scipy.stats import norm
from sklearn.metrics import roc_auc_score
from pathlib import Path
from scipy.spatial.distance import pdist
from tqdm.notebook import trange, tqdm

sys.path.append('/om2/user/jmhicks/projects/TextureStreaming/code/')
sys.path.append('/om2/user/bjmedina/auditory-memory/memory/utls/')
sys.path.append('/om2/user/bjmedina/auditory-memory/memory/src/model/')
sys.path.append('/om2/user/bjmedina/auditory-memory/memory/')

from chexture_choolbox.auditorytexture.texture_model import TextureModel
from chexture_choolbox.auditorytexture.helpers import FlattenStats
from texture_prior.params import model_params, statistics_set, texture_dataset
from texture_prior.utils import path
from utls.plotting import ensure_dir
from utls.loading import (load_results_with_exclusion_2, move_sequences_to_used,
                           load_results_with_exclusion_no_dropping)
from utls.runners_v2 import run_experiment_scores, make_noise_schedule
from utls.runners_utils import *
from encoders import *
from utls.toy_experiments import (
    make_isi_n_block_experiment,
    make_toy_experiment_list,
    make_multi_isi_toy_experiments,
    make_compact_multi_isi_sequences,
    infer_trial_isis,
)
from utls.sigma_fitting import (
    log_mid, make_grid, auc_to_dprime,
    evaluate_sigma_on_multi_isi_sequences,
)

In [ ]:
def load_config(cfg_path):
    cfg_path = Path(cfg_path)
    if not cfg_path.exists():
        raise FileNotFoundError(cfg_path)
    with open(cfg_path) as f:
        return yaml.safe_load(f), cfg_path

def median_pairwise_distance(X, metric='euclidean', n_samples=500, seed=0):
    rng = np.random.default_rng(seed)
    idx = rng.choice(X.shape[0], size=min(n_samples, X.shape[0]), replace=False)
    return float(np.median(pdist(X[idx], metric=metric)))

CONFIG_PATH = (
    '/om2/user/bjmedina/auditory-memory/memory/'
    'model_yamls/three-regime/resnet50/nontime_avg/run_000005.yaml'
)
model_cfg, model_cfg_path = load_config(CONFIG_PATH)
print(model_cfg)

In [ ]:
exp_cfg    = model_cfg['experiment']
which_task = exp_cfg['which_task']
is_multi   = exp_cfg['is_multi']
which_isi  = exp_cfg.get('which_isi', None)
isis       = [0, 1, 2, 4, 8, 16, 32, 64] if is_multi else [0, which_isi]
metric     = model_cfg['metric']
noise_cfg  = model_cfg['noise_model']
noise_mode = noise_cfg['name']
t_step     = noise_cfg['t_step']
repr_cfg   = model_cfg['representation']
time_avg   = repr_cfg['time_avg']
encoder_type = repr_cfg['type']
layer      = repr_cfg.get('layer', None)
pc_dims    = repr_cfg.get('pc_dims', None)

exp_list, all_files, name_to_idx, human_runs, task_name, hr_task_name = (
    load_experiment_data(which_task, which_isi, is_multi, old=False)
)
human_curve  = compute_human_curve(human_runs, is_multi, which_isi)
time_avg_tag = 'time_avg' if time_avg else 'nontime_avg'
print('ISIs:', isis)
print('Human d\':', human_curve)
print(f'Total real sequences: {len(exp_list)}')

In [ ]:
NN_ENCODERS  = {'kell2018', 'resnet50'}
encoder_task = 'word_speaker_audioset' if encoder_type in NN_ENCODERS else 'audioset'
encoder_cfg  = dict(
    encoder_type=encoder_type, model_name=encoder_type, task=encoder_task,
    statistics_dict=statistics_set.statistics, model_params=model_params,
    pc_dims=pc_dims, sr=20000, duration=2.0, rms_level=0.05,
    time_avg=time_avg, device='cuda',
)
if encoder_type in NN_ENCODERS: encoder_cfg['layer'] = layer
if encoder_type == 'texture':   encoder_cfg['pc_dims'] = pc_dims

encoder_name = make_encoder_name(encoder_cfg)
encoder      = build_encoder(encoder_cfg)
X            = encode_stimuli(encoder, all_files)
X_np         = X.detach().cpu().numpy()
print('Shape:', X_np.shape, ' NaN?', torch.isnan(X).any().item())

d50 = median_pairwise_distance(X_np, metric='cosine')
print(f'd50 = {d50:.6f}')

param_bounds = {
    'sigma0': (noise_cfg['sigma0_min'],         noise_cfg['sigma0_max']),
    'sigma1': (noise_cfg['sigma1_min'] * d50,   noise_cfg['sigma1_max'] * d50),
    'sigma2': (noise_cfg['sigma2_min'] * d50,   noise_cfg['sigma2_max'] * d50),
}
for k, v in param_bounds.items():
    print(f'  {k}: ({v[0]:.6f}, {v[1]:.6f})')

stimulus_pool = sorted({s for seq in exp_list for s in seq})
print(f'Stimulus pool: {len(stimulus_pool)}')

In [ ]:
sigma0_fixed = log_mid(*param_bounds['sigma0'])
sigma1_fixed = log_mid(*param_bounds['sigma1'])
print(f'Fixed sigma0 = {sigma0_fixed:.6f}')
print(f'Fixed sigma1 = {sigma1_fixed:.6f}')

isi_to_hc_idx = {isi_val: i for i, isi_val in enumerate(isis)}

TARGET_ISIS = [8, 16, 32, 64]
human_dprimes_by_isi = {}
for isi_val in TARGET_ISIS:
    idx = isi_to_hc_idx.get(isi_val)
    if idx is not None:
        human_dprimes_by_isi[isi_val] = human_curve[idx]
        print(f'  ISI {isi_val}: human d\' = {human_curve[idx]:.4f}')

N_PER_DIM = 8
sigma2_grid = make_grid(param_bounds['sigma2'][0], param_bounds['sigma2'][1],
                        N_PER_DIM, spacing='log')
print('\nsigma2 grid:', [f'{v:.6f}' for v in sigma2_grid])

## Section A: Generate & Validate Compact Multi-ISI Sequences

Generate sequences using `ISISequence` + `StimulusManager` with only ISIs
[8, 16, 32, 64]. Each sequence is ~78 trials with ~5-6 pairs per ISI.
Compare to per-ISI toy experiments that would need ~130 trials each for ISI=64.

In [ ]:
# Generate compact multi-ISI sequences
COMPACT_LENGTH = 78       # divisible by 3, fits ISI=64
N_COMPACT_SEQS = 30       # enough to sample subsets per MC rep
MIN_PAIRS      = 5        # target ~5 pairs per ISI (matches real experiments)

compact_exps, compact_isi_keys = make_compact_multi_isi_sequences(
    stimulus_pool=stimulus_pool,
    isi_values=TARGET_ISIS,
    n_sequences=N_COMPACT_SEQS,
    length=COMPACT_LENGTH,
    min_pairs_per_isi=MIN_PAIRS,
    seed=42,
)

print(f'Generated {len(compact_exps)} compact sequences')
print(f'Trials per sequence: {len(compact_exps[0])}')
print(f'\nComparison:')
print(f'  Per-ISI toy (ISI=64): ~{2*(64+1)} trials per experiment')
print(f'  Compact multi-ISI:    ~{COMPACT_LENGTH} trials per sequence')

In [ ]:
# Validate: check ISI distribution across generated sequences
all_isi_counts = defaultdict(list)

for i, (seq, isi_key) in enumerate(zip(compact_exps, compact_isi_keys)):
    trial_isis = infer_trial_isis(seq)
    counts = Counter(trial_isis)
    for isi_val in TARGET_ISIS:
        all_isi_counts[isi_val].append(counts.get(isi_val, 0))

print('Pairs per ISI per sequence (mean +/- std):')
for isi_val in TARGET_ISIS:
    vals = all_isi_counts[isi_val]
    print(f'  ISI {isi_val:>2}: {np.mean(vals):.1f} +/- {np.std(vals):.1f}  '
          f'(min={min(vals)}, max={max(vals)})')

# Total trial counts
total_repeat_trials = sum(np.sum(v) for v in all_isi_counts.values())
total_trials = sum(len(s) for s in compact_exps)
print(f'\nTotal repeat trials across all sequences: {total_repeat_trials}')
print(f'Total trials across all sequences: {total_trials}')

In [ ]:
# Visualize ISI distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: pairs per ISI per sequence
ax = axes[0]
for isi_val in TARGET_ISIS:
    ax.plot(all_isi_counts[isi_val], 'o-', markersize=3, label=f'ISI {isi_val}')
ax.set_xlabel('Sequence index')
ax.set_ylabel('Repeat pairs')
ax.set_title('Repeat pairs per ISI per sequence')
ax.legend(); ax.grid(alpha=0.3)

# Right: histogram of pair counts
ax = axes[1]
for isi_val in TARGET_ISIS:
    ax.hist(all_isi_counts[isi_val], bins=range(0, 15), alpha=0.5, label=f'ISI {isi_val}')
ax.set_xlabel('Pairs per sequence')
ax.set_ylabel('Count')
ax.set_title('Distribution of pair counts')
ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout(); plt.show()

## Section B: Sigma2 Grid Sweep with Compact Sequences

Sweep sigma2 values and compute MSE vs human d' targets.
Uses `evaluate_sigma_on_multi_isi_sequences` which handles
multi-ISI hit splitting internally.

In [ ]:
N_MC          = 32
N_SEQS_PER_REP = 10  # sample 10 of 30 sequences per MC rep

compact_sweep_results = []
for i, sigma_val in enumerate(tqdm(sigma2_grid, desc='Sigma2 sweep')):
    result = evaluate_sigma_on_multi_isi_sequences(
        run_experiment_fn=run_experiment_scores,
        sigma_value=sigma_val,
        sigma_name='sigma2',
        fixed_sigmas={'sigma0': sigma0_fixed, 'sigma1': sigma1_fixed},
        noise_mode=noise_mode,
        metric=metric,
        X0=X,
        name_to_idx=name_to_idx,
        experiment_list=compact_exps,
        isi_keys=compact_isi_keys,
        target_isis=TARGET_ISIS,
        human_dprimes_by_isi=human_dprimes_by_isi,
        t_step=t_step,
        n_seqs_per_rep=N_SEQS_PER_REP,
        n_mc=N_MC,
        seed=i * 100_000,
    )
    compact_sweep_results.append(result)
    print(f'  sigma2={sigma_val:.6f}  MSE={result["mse_mean"]:.6f} +/- {result["mse_std"]:.4f}')

print('\nDone.')

In [ ]:
df_compact = pd.DataFrame(compact_sweep_results)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: MSE landscape
ax = axes[0]
ax.plot(df_compact.sigma_value, df_compact.mse_mean, 'o-', color='C0')
ax.fill_between(df_compact.sigma_value,
                df_compact.mse_mean - df_compact.mse_std,
                df_compact.mse_mean + df_compact.mse_std, alpha=0.2)
best_idx = df_compact.mse_mean.idxmin()
best_sigma = df_compact.loc[best_idx, 'sigma_value']
ax.axvline(best_sigma, ls='--', color='red', alpha=0.6,
           label=f'best={best_sigma:.4f}')
ax.set_xscale('log'); ax.set_xlabel(r'$\sigma_2$'); ax.set_ylabel('MSE')
ax.set_title(f'Sigma2 MSE landscape (compact, N_seqs={N_SEQS_PER_REP}, N_MC={N_MC})')
ax.legend(); ax.grid(alpha=0.3)

# Right: d' per ISI
ax = axes[1]
for isi_val in TARGET_ISIS:
    dp_means = [r['dprime_mean_by_isi'][isi_val] for r in compact_sweep_results]
    dp_stds  = [r['dprime_std_by_isi'][isi_val] for r in compact_sweep_results]
    ax.errorbar(df_compact.sigma_value, dp_means, yerr=dp_stds,
                fmt='o-', capsize=3, markersize=3, label=f'ISI {isi_val}')
    ax.axhline(human_dprimes_by_isi[isi_val], ls=':', alpha=0.4,
               label=f'human ISI={isi_val}')
ax.axvline(best_sigma, ls='--', color='red', alpha=0.6)
ax.set_xscale('log'); ax.set_xlabel(r'$\sigma_2$'); ax.set_ylabel("d'")
ax.set_title("Model d' vs sigma2 (compact sequences)")
ax.legend(fontsize=7, ncol=2); ax.grid(alpha=0.3)

plt.tight_layout(); plt.show()
print(f'\nBest sigma2 = {best_sigma:.6f} (MSE = {df_compact.loc[best_idx, "mse_mean"]:.6f})')

## Section C: Stability — Vary N Sequences per Rep

How many compact sequences per MC rep are needed for a stable sigma2 estimate?
Real experiments have ~5 trials per ISI per sequence, so:
- 1 seq  -> ~5 trials per ISI
- 5 seqs -> ~25 trials per ISI
- 10 seqs -> ~50 trials per ISI
- 20 seqs -> ~100 trials per ISI

In [ ]:
SEQ_COUNTS = [1, 5, 10, 20]
stability_results = {}

for n_seq in SEQ_COUNTS:
    print(f'\n--- n_seqs_per_rep={n_seq} ---')
    results = []
    for i, sigma_val in enumerate(tqdm(sigma2_grid, desc=f'N={n_seq}', leave=False)):
        result = evaluate_sigma_on_multi_isi_sequences(
            run_experiment_fn=run_experiment_scores,
            sigma_value=sigma_val,
            sigma_name='sigma2',
            fixed_sigmas={'sigma0': sigma0_fixed, 'sigma1': sigma1_fixed},
            noise_mode=noise_mode,
            metric=metric,
            X0=X,
            name_to_idx=name_to_idx,
            experiment_list=compact_exps,
            isi_keys=compact_isi_keys,
            target_isis=TARGET_ISIS,
            human_dprimes_by_isi=human_dprimes_by_isi,
            t_step=t_step,
            n_seqs_per_rep=n_seq,
            n_mc=N_MC,
            seed=300_000_000 + n_seq * 1_000_000 + i * 100_000,
        )
        results.append(result)
    stability_results[n_seq] = results

print('\nDone.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: MSE landscape by N
ax = axes[0]
for n_seq, results in stability_results.items():
    df = pd.DataFrame(results)
    ax.plot(df.sigma_value, df.mse_mean, 'o-', label=f'N={n_seq}')
    ax.fill_between(df.sigma_value, df.mse_mean - df.mse_std,
                    df.mse_mean + df.mse_std, alpha=0.15)
ax.set_xscale('log'); ax.set_xlabel(r'$\sigma_2$'); ax.set_ylabel('MSE')
ax.set_title(f'Sigma2 stability by N_seqs (compact, N_MC={N_MC})')
ax.legend(); ax.grid(alpha=0.3)

# Right: best sigma2 and its uncertainty per N
ax = axes[1]
best_sigmas = []
for n_seq, results in stability_results.items():
    df = pd.DataFrame(results)
    best_i = df.mse_mean.idxmin()
    best_sigmas.append({
        'n_seq': n_seq,
        'best_sigma2': df.loc[best_i, 'sigma_value'],
        'best_mse': df.loc[best_i, 'mse_mean'],
        'mse_std': df.loc[best_i, 'mse_std'],
    })
df_best = pd.DataFrame(best_sigmas)
ax.errorbar(df_best.n_seq, df_best.best_sigma2, fmt='o-', capsize=5, markersize=8)
ax.set_xlabel('N sequences per rep'); ax.set_ylabel(r'Best $\sigma_2$')
ax.set_title('Best sigma2 vs N sequences')
ax.grid(alpha=0.3)

plt.tight_layout(); plt.show()
print(df_best.to_string(index=False))

## Section D: Comparison — Compact vs Per-ISI Toy Baseline

Run the same sigma2 sweep using traditional per-ISI toy experiments
(k_stimuli = ISI + 2, n_experiments = 20) as a reference.

In [ ]:
from utls.sigma_fitting import evaluate_sigma_on_toy_experiments

# Generate per-ISI toy experiments for comparison
N_TOY_EXPS = 20
toy_exps_by_isi = {}
for isi_val in TARGET_ISIS:
    k_stim = min(isi_val + 2, len(stimulus_pool))
    exps = make_toy_experiment_list(
        stimulus_pool, isi=isi_val, n_experiments=N_TOY_EXPS,
        k_stimuli=k_stim, seed=isi_val * 1000 + N_TOY_EXPS)
    toy_exps_by_isi[isi_val] = [e for e in exps if e]
    avg_len = np.mean([len(e) for e in toy_exps_by_isi[isi_val]])
    print(f'ISI {isi_val:>2}: {len(toy_exps_by_isi[isi_val])} exps, avg len {avg_len:.1f}')

total_toy_trials = sum(
    sum(len(e) for e in exps) for exps in toy_exps_by_isi.values()
)
total_compact_trials = sum(len(s) for s in compact_exps[:N_SEQS_PER_REP])
print(f'\nTotal trials (toy, all ISIs): {total_toy_trials}')
print(f'Total trials (compact, {N_SEQS_PER_REP} seqs): {total_compact_trials}')
print(f'Reduction: {total_toy_trials / total_compact_trials:.1f}x')

In [ ]:
toy_sweep_results = []
for i, sigma_val in enumerate(tqdm(sigma2_grid, desc='Toy baseline')):
    result = evaluate_sigma_on_toy_experiments(
        run_experiment_fn=run_experiment_scores,
        sigma_value=sigma_val,
        sigma_name='sigma2',
        fixed_sigmas={'sigma0': sigma0_fixed, 'sigma1': sigma1_fixed},
        noise_mode=noise_mode,
        metric=metric,
        X0=X,
        name_to_idx=name_to_idx,
        experiments_by_isi=toy_exps_by_isi,
        human_dprimes_by_isi=human_dprimes_by_isi,
        t_step=t_step,
        n_mc=N_MC,
        seed=400_000_000 + i * 100_000,
    )
    toy_sweep_results.append(result)

print('Done.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: MSE comparison
ax = axes[0]
df_toy = pd.DataFrame([{
    'sigma_value': r['sigma_value'],
    'mse_mean': r['mse_mean'],
} for r in toy_sweep_results])
ax.plot(df_compact.sigma_value, df_compact.mse_mean, 'o-', label='Compact multi-ISI')
ax.plot(df_toy.sigma_value, df_toy.mse_mean, 's--', label='Per-ISI toy', alpha=0.7)
ax.set_xscale('log'); ax.set_xlabel(r'$\sigma_2$'); ax.set_ylabel('MSE')
ax.set_title('MSE landscape: compact vs per-ISI toy')
ax.legend(); ax.grid(alpha=0.3)

# Right: d' comparison per ISI (at best compact sigma2)
ax = axes[1]
best_compact = compact_sweep_results[best_idx]
best_toy_idx = df_toy.mse_mean.idxmin()
best_toy = toy_sweep_results[best_toy_idx]

x_pos = np.arange(len(TARGET_ISIS))
width = 0.25
human_dps = [human_dprimes_by_isi[isi] for isi in TARGET_ISIS]
compact_dps = [best_compact['dprime_mean_by_isi'][isi] for isi in TARGET_ISIS]
toy_dps = [best_toy['dprime_by_isi'].get(isi, np.nan) for isi in TARGET_ISIS]

ax.bar(x_pos - width, human_dps, width, label='Human', color='gray', alpha=0.7)
ax.bar(x_pos, compact_dps, width, label=f'Compact (s2={best_sigma:.4f})')
ax.bar(x_pos + width, toy_dps, width, label=f'Toy (s2={df_toy.loc[best_toy_idx, "sigma_value"]:.4f})', alpha=0.7)
ax.set_xticks(x_pos)
ax.set_xticklabels([f'ISI {isi}' for isi in TARGET_ISIS])
ax.set_ylabel("d'"); ax.set_title("d' at best sigma2: compact vs toy")
ax.legend(); ax.grid(alpha=0.3, axis='y')

plt.tight_layout(); plt.show()

## Summary

*(Fill in after running)*

**Key findings:**
- Best sigma2 from compact sequences: 
- Best sigma2 from per-ISI toy baseline: 
- Agreement between methods: 
- Minimum sequences for stable estimate: 
- Trial reduction factor: 